In [1]:
import pandas as pd

# Define the path to your subsidy file
subsidy_file = "agrisub-2026-04-08.csv"

print("Inspecting the subsidy dataset (skipping the first row)...")

# Use skiprows=1 to ignore the metadata line at the top
df_sub_preview = pd.read_csv(subsidy_file, sep=";", skiprows=1, nrows=5, dtype=str)

# 1. Print all exact column names
print("\n--- Exact Column Names ---")
columns = df_sub_preview.columns.tolist()
for col in columns:
    print(f"- '{col}'")

# 2. Print a small preview to visually confirm the data
print("\n--- Data Preview ---")
print(df_sub_preview.head())

Inspecting the subsidy dataset (skipping the first row)...

--- Exact Column Names ---
- 'Nom attributaire*'
- 'Identification de l'attributaire*'
- 'Date de convention*'
- 'Référence de la décision'
- 'Identification du bénéficiaire*'
- 'Nom du bénéficiaire*'
- 'Objet de la convention'
- 'Montant total de la subvention*'
- 'Nature de la subvention*'
- 'Conditions de versement*'
- 'Date de versement'
- 'Numéro de référencement au répertoire des entreprises'
- 'Aide notifiée à l'Europe'
- 'Pourcentage du montant de la subvention attribué au bénéficiaire*'

--- Data Preview ---
                                   Nom attributaire*  \
0  MAA - Ministère de l'Agriculture et de l'Alime...   
1  MAA - Ministère de l'Agriculture et de l'Alime...   
2  MAA - Ministère de l'Agriculture et de l'Alime...   
3  MAA - Ministère de l'Agriculture et de l'Alime...   
4  MAA - Ministère de l'Agriculture et de l'Alime...   

  Identification de l'attributaire* Date de convention*  \
0                11 0

In [2]:
import pandas as pd

# Define file paths
sirene_file = "StockUniteLegale.csv"
subsidy_file = "agrisub-2026-04-08.csv" # Adjust to your actual filename

# 1. Load Subsidy Data
print("Loading subsidy data...")
# SCDL files usually use ';' as separator, but adjust if needed
df_sub = pd.read_csv(subsidy_file, sep=";", skiprows=1, dtype=str) 

# Ensure column name matches exactly what's in the CSV (often 'id_beneficiaire' or 'siret'/'siren' in SCDL)
# For this example, assuming the column is named 'id_beneficiaire'
sub_siren_col = 'Identification du bénéficiaire*' 

# Extract SIREN from SIRET if the dataset provides SIRETs (14 chars) instead of SIRENs (9 chars)
if df_sub[sub_siren_col].str.len().max() == 14:
    df_sub['siren'] = df_sub[sub_siren_col].str[:9]
else:
    df_sub['siren'] = df_sub[sub_siren_col]

# Get unique SIRENs that received a subsidy to filter Sirene efficiently
subsidized_sirens = set(df_sub['siren'].dropna())
print(f"Found {len(subsidized_sirens)} unique subsidized SIRENs.")

Loading subsidy data...
Found 137 unique subsidized SIRENs.


In [3]:
# 2. Process Sirene Data in Chunks (Crucial for large files)
print("\nProcessing Sirene data...")

chunk_size = 100000 
filtered_sirene_list = []

# We use an iterator to read the massive file piece by piece
for chunk in pd.read_csv(sirene_file, chunksize=chunk_size, dtype=str):
    
    # Filter 1: Keep only agricultural sectors (codes starting with '01' or '02')
    # Use fillna('') to avoid errors on empty values
    is_agri = chunk['activitePrincipaleUniteLegale'].fillna('').str.startswith(('01', '02'))
    
    # Filter 2: Keep only SIRENs present in our subsidy dataset
    in_subsidy = chunk['siren'].isin(subsidized_sirens)
    
    # Apply both filters
    filtered_chunk = chunk[is_agri | in_subsidy] 
    
    if not filtered_chunk.empty:
        filtered_sirene_list.append(filtered_chunk)

# Combine all valid chunks into one DataFrame
df_sirene_filtered = pd.concat(filtered_sirene_list, ignore_index=True)

# Keep only essential columns to save memory
columns_to_keep = [
    'siren', 
    'nomUniteLegale', 
    'denominationUniteLegale', 
    'activitePrincipaleUniteLegale',
    'categorieEntreprise'
]
df_sirene_filtered = df_sirene_filtered[columns_to_keep]

print(f"Filtered Sirene dataset size: {len(df_sirene_filtered)} companies.")


Processing Sirene data...
Filtered Sirene dataset size: 1535035 companies.


In [4]:
# 3. Perform the Join with correct column names
print("\nMerging datasets...")

# Merge based on the common 'siren' column
df_merged = pd.merge(df_sub, df_sirene_filtered, on='siren', how='inner')

print(f"Merge successful! Resulting dataset has {len(df_merged)} records.")

# Define the exact columns we want to inspect in the preview
preview_columns = [
    'siren', 
    'Nom du bénéficiaire*', 
    'Montant total de la subvention*', 
    'activitePrincipaleUniteLegale'
]

# Preview the result
print("\n--- First 5 rows of merged data ---")
print(df_merged[preview_columns].head())


Merging datasets...
Merge successful! Resulting dataset has 0 records.

--- First 5 rows of merged data ---
Empty DataFrame
Columns: [siren, Nom du bénéficiaire*, Montant total de la subvention*, activitePrincipaleUniteLegale]
Index: []


In [5]:
import pandas as pd

subsidy_file = "agrisub-2026-04-08.csv"
df_sub = pd.read_csv(subsidy_file, sep=";", skiprows=1, dtype=str)
id_col = "Identification du bénéficiaire*"

print("--- 1. Échantillon brut (les 10 premiers IDs) ---")
# On transforme en liste pour bien voir s'il y a des espaces
print(df_sub[id_col].dropna().head(10).tolist())

print("\n--- 2. Tailles des identifiants trouvés ---")
# Cela va nous dire si on a des nombres à 9 chiffres, 14 chiffres, ou autre chose
print(df_sub[id_col].dropna().str.len().value_counts())

print("\n--- 3. Échantillon des noms (Plan B) ---")
print(df_sub['Nom du bénéficiaire*'].dropna().head(5).tolist())

--- 1. Échantillon brut (les 10 premiers IDs) ---
['78\u202f452\u202f331\u202f800\u202f011', '30\u202f298\u202f415\u202f800\u202f022', '30\u202f298\u202f415\u202f800\u202f022', '77\u202f568\u202f577\u202f900\u202f313', '30\u202f298\u202f415\u202f800\u202f022', '77\u202f568\u202f223\u202f000\u202f062', '78\u202f452\u202f331\u202f800\u202f011', '78\u202f452\u202f331\u202f800\u202f011', '78\u202f452\u202f331\u202f800\u202f011', '77\u202f567\u202f579\u202f600\u202f012']

--- 2. Tailles des identifiants trouvés ---
Identification du bénéficiaire*
18    650
17      5
13      1
Name: count, dtype: int64

--- 3. Échantillon des noms (Plan B) ---
['ACTA', "Instituit de l'Elevage (IDELE)", "Instituit de l'Elevage (IDELE)", 'ARVALIS', "Instituit de l'Elevage (IDELE)"]


In [6]:
import pandas as pd

# Tes fichiers
sirene_file = "StockUniteLegale.csv"
subsidy_file = "agrisub-2026-04-08.csv"

# ---------------------------------------------------------
# 1. Chargement et nettoyage des subventions
# ---------------------------------------------------------
print("Loading and cleaning subsidy data...")
df_sub = pd.read_csv(subsidy_file, sep=";", skiprows=1, dtype=str) 

sub_id_col = "Identification du bénéficiaire*"

# LA CORRECTION EST ICI : 
# .str.replace(r'\s+', '', regex=True) supprime TOUS les types d'espaces
# .str[:9] récupère ensuite les 9 premiers chiffres (le SIREN)
df_sub['siren'] = df_sub[sub_id_col].fillna('').str.replace(r'\s+', '', regex=True).str[:9]

subsidized_sirens = set(df_sub['siren'])
subsidized_sirens.discard('') # On enlève les valeurs vides

print(f"Found {len(subsidized_sirens)} unique clean SIRENs.")


# ---------------------------------------------------------
# 2. Traitement de la base Sirene
# ---------------------------------------------------------
print("\nProcessing Sirene data in chunks (this will take a few minutes)...")

chunk_size = 100000 
filtered_sirene_list = []
chunk_number = 0

for chunk in pd.read_csv(sirene_file, chunksize=chunk_size, dtype=str):
    chunk_number += 1
    print(f"Traitement du bloc {chunk_number}...", end='\r')
    
    # Filtres
    is_agri = chunk['activitePrincipaleUniteLegale'].fillna('').str.startswith(('01', '02'))
    in_subsidy = chunk['siren'].isin(subsidized_sirens)
    
    filtered_chunk = chunk[is_agri | in_subsidy] 
    
    if not filtered_chunk.empty:
        filtered_sirene_list.append(filtered_chunk)

print("\nCombining filtered Sirene data...")
df_sirene_filtered = pd.concat(filtered_sirene_list, ignore_index=True)

columns_to_keep = [
    'siren', 
    'nomUniteLegale', 
    'denominationUniteLegale', 
    'activitePrincipaleUniteLegale',
    'categorieEntreprise'
]
df_sirene_filtered = df_sirene_filtered[columns_to_keep]


# ---------------------------------------------------------
# 3. La Jointure
# ---------------------------------------------------------
print("\nMerging datasets...")
df_merged = pd.merge(df_sub, df_sirene_filtered, on='siren', how='inner')

print(f"Merge successful! Resulting dataset has {len(df_merged)} records.")

preview_columns = [
    'siren', 
    'Nom du bénéficiaire*', 
    'Montant total de la subvention*', 
    'activitePrincipaleUniteLegale'
]

print("\n--- First 5 rows of merged data ---")
print(df_merged[preview_columns].head())

Loading and cleaning subsidy data...
Found 125 unique clean SIRENs.

Processing Sirene data in chunks (this will take a few minutes)...
Traitement du bloc 296...
Combining filtered Sirene data...

Merging datasets...
Merge successful! Resulting dataset has 651 records.

--- First 5 rows of merged data ---
       siren            Nom du bénéficiaire* Montant total de la subvention*  \
0  784523318                            ACTA                          60 000   
1  302984158  Instituit de l'Elevage (IDELE)                          60 000   
2  302984158  Instituit de l'Elevage (IDELE)                          60 000   
3  775685779                         ARVALIS                         120 000   
4  302984158  Instituit de l'Elevage (IDELE)                          60 000   

  activitePrincipaleUniteLegale  
0                        72.19Z  
1                        72.19Z  
2                        72.19Z  
3                        72.19Z  
4                        72.19Z  


In [7]:
import pandas as pd

print("Création du Knowledge Graph (Triplets)...")

triplets = []

# On parcourt chaque ligne de notre tableau fusionné
for index, row in df_merged.iterrows():
    # 1. Définition des Nœuds (On ajoute des préfixes pour bien les différencier dans le graphe)
    nœud_entreprise = f"ENTREPRISE_{row['siren']}"
    nœud_secteur = f"SECTEUR_{row['activitePrincipaleUniteLegale']}"
    nœud_financeur = "MINISTERE_AGRICULTURE"
    
    # 2. Création des relations (Sujet, Relation, Objet)
    
    # L'entreprise opère dans un secteur
    triplets.append([nœud_entreprise, "OPERE_DANS", nœud_secteur])
    
    # L'entreprise a reçu une subvention du ministère
    triplets.append([nœud_entreprise, "A_RECU_SUBVENTION_DE", nœud_financeur])
    
    # --- Astuce KGE (Knowledge Graph Embeddings) ---
    # TransE a du mal avec les valeurs numériques exactes (comme 60000€).
    # Il est préférable de créer des "catégories" (nœuds) pour les montants.
    montant_str = str(row['Montant total de la subvention*']).replace(' ', '').replace(' ', '')
    try:
        montant = float(montant_str)
        if montant > 100000:
            nœud_montant = "TRANCHE_SUPERIEURE_100K"
        elif montant > 50000:
            nœud_montant = "TRANCHE_50K_100K"
        else:
            nœud_montant = "TRANCHE_INFERIEURE_50K"
            
        triplets.append([nœud_entreprise, "A_RECU_MONTANT", nœud_montant])
    except ValueError:
        pass # Si le montant est vide ou illisible, on passe

# On transforme notre liste en DataFrame formaté pour PyKEEN
df_kg = pd.DataFrame(triplets, columns=["head", "relation", "tail"])

print(f"Le Knowledge Graph a été créé avec succès ! Il contient {len(df_kg)} triplets (liens).")

print("\n--- Aperçu des 5 premiers triplets du Graphe ---")
print(df_kg.head(5))

# Sauvegarde du graphe pour la suite
df_kg.to_csv("mon_knowledge_graph.tsv", sep='\t', index=False)
print("\nGraphe sauvegardé sous le nom 'mon_knowledge_graph.tsv'")

Création du Knowledge Graph (Triplets)...
Le Knowledge Graph a été créé avec succès ! Il contient 1939 triplets (liens).

--- Aperçu des 5 premiers triplets du Graphe ---
                   head              relation                   tail
0  ENTREPRISE_784523318            OPERE_DANS         SECTEUR_72.19Z
1  ENTREPRISE_784523318  A_RECU_SUBVENTION_DE  MINISTERE_AGRICULTURE
2  ENTREPRISE_784523318        A_RECU_MONTANT       TRANCHE_50K_100K
3  ENTREPRISE_302984158            OPERE_DANS         SECTEUR_72.19Z
4  ENTREPRISE_302984158  A_RECU_SUBVENTION_DE  MINISTERE_AGRICULTURE

Graphe sauvegardé sous le nom 'mon_knowledge_graph.tsv'


In [8]:
import pandas as pd

print("Lancement du raisonneur logique...")

nouveaux_triplets = []

# --- RÈGLE LOGIQUE 1 : Détection des "Bénéficiaires Multiples" ---
# Si une entreprise a reçu plus d'UNE subvention, on la marque comme "BENEFICIAIRE_MULTIPLE"

# On compte combien de fois chaque entreprise apparaît avec la relation "A_RECU_SUBVENTION_DE"
comptage_subventions = df_kg[df_kg['relation'] == 'A_RECU_SUBVENTION_DE'].groupby('head').size()
beneficiaires_multiples = comptage_subventions[comptage_subventions > 1].index.tolist()

for entreprise in beneficiaires_multiples:
    nouveaux_triplets.append([entreprise, "A_LE_STATUT_DE", "BENEFICIAIRE_MULTIPLE"])

# --- RÈGLE LOGIQUE 2 : Détection des "Instituts de Recherche Agricole" ---
# Si une entreprise opère dans le secteur de la recherche (72.19Z) ET a reçu de l'argent de l'Agriculture
# Alors c'est un "ACTEUR_RECHERCHE_AGRI"

# On cherche les entreprises dans le secteur 72.19Z
entreprises_rd = df_kg[(df_kg['relation'] == 'OPERE_DANS') & (df_kg['tail'] == 'SECTEUR_72.19Z')]['head'].tolist()

for entreprise in entreprises_rd:
    # Pas besoin de revérifier la subvention agricole ici car toutes les entreprises 
    # de notre graphe actuel en ont reçu une (filtre de la Phase 1)
    nouveaux_triplets.append([entreprise, "EST_UN", "ACTEUR_RECHERCHE_AGRI"])

# --- Intégration des nouvelles connaissances au graphe ---
if nouveaux_triplets:
    df_inferences = pd.DataFrame(nouveaux_triplets, columns=["head", "relation", "tail"])
    
    # On fusionne l'ancien graphe avec les nouvelles déductions
    df_kg_enrichi = pd.concat([df_kg, df_inferences], ignore_index=True)
    
    print(f"Raisonnement terminé ! {len(nouveaux_triplets)} nouveaux triplets ont été déduits logiquement.")
    print(f"Taille du graphe enrichi : {len(df_kg_enrichi)} triplets.")
    
    # Sauvegarde du graphe final enrichi
    df_kg_enrichi.to_csv("knowledge_graph_enrichi.tsv", sep='\t', index=False)
    print("Graphe enrichi sauvegardé sous 'knowledge_graph_enrichi.tsv'")
    
    print("\n--- Aperçu des nouvelles connaissances (Inférences) ---")
    print(df_inferences.head(5))
else:
    print("Aucune nouvelle connaissance n'a pu être déduite avec ces règles.")

Lancement du raisonneur logique...
Raisonnement terminé ! 412 nouveaux triplets ont été déduits logiquement.
Taille du graphe enrichi : 2351 triplets.
Graphe enrichi sauvegardé sous 'knowledge_graph_enrichi.tsv'

--- Aperçu des nouvelles connaissances (Inférences) ---
                   head        relation                   tail
0  ENTREPRISE_130006364  A_LE_STATUT_DE  BENEFICIAIRE_MULTIPLE
1  ENTREPRISE_130021660  A_LE_STATUT_DE  BENEFICIAIRE_MULTIPLE
2  ENTREPRISE_302984158  A_LE_STATUT_DE  BENEFICIAIRE_MULTIPLE
3  ENTREPRISE_320387889  A_LE_STATUT_DE  BENEFICIAIRE_MULTIPLE
4  ENTREPRISE_325346542  A_LE_STATUT_DE  BENEFICIAIRE_MULTIPLE


In [9]:
import pandas as pd
import torch
import torch.nn.functional as F
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

print("1. Loading the Knowledge Graph into PyKEEN...")
# Load the TSV file we created in Phase 3
tf = TriplesFactory.from_path("knowledge_graph_enrichi.tsv")

print(f"Graph loaded! Number of entities: {tf.num_entities}, Number of relations: {tf.num_relations}")

print("\n2. Training the TransE Embedding Model (This may take 1-2 minutes)...")
# We use the pipeline to train TransE. 
# We use 50 epochs to keep it fast for this mini-project.
result = pipeline(
    training=tf,
    testing=tf, # For demonstration, we test on the same graph
    model='TransE',
    training_kwargs=dict(num_epochs=50),
    random_seed=42,
    device='cpu' # Use 'cuda' if you have an Nvidia GPU setup
)

print("\nTraining complete! Extracting embeddings...")
model = result.model
# Extract the dense vectors (embeddings) for all entities
entity_embeddings = model.entity_representations[0](indices=None).detach()

# ---------------------------------------------------------
# 3. Task: Find the most similar companies (Embeddings in action!)
# ---------------------------------------------------------
print("\n3. Analyzing similarities based on graph structure...")

# Let's take IDELE as an example (we saw its SIREN '302984158' in your previous outputs)
target_entity_name = 'ENTREPRISE_302984158'

# 1. Find the internal PyKEEN ID for this entity
entity_to_id = tf.entity_to_id
if target_entity_name in entity_to_id:
    target_id = entity_to_id[target_entity_name]
    target_vector = entity_embeddings[target_id]
    
    # 2. Compute Cosine Similarity between this company and ALL other entities
    # Cosine similarity is a math function that returns 1 for identical vectors, and 0 or less for different ones
    similarities = F.cosine_similarity(target_vector.unsqueeze(0), entity_embeddings)
    
    # 3. Get the top 5 most similar entities (excluding itself)
    top_k = 6 # We take 6 because the 1st one will be the entity itself
    top_scores, top_indices = torch.topk(similarities, top_k)
    
    print(f"\n--- Top 5 entities most similar to {target_entity_name} ---")
    id_to_entity = {v: k for k, v in entity_to_id.items()} # Reverse dictionary mapping
    
    for score, idx in zip(top_scores, top_indices):
        idx = idx.item()
        entity_name = id_to_entity[idx]
        if entity_name != target_entity_name:
            # We only print it if it's actually another company
            if entity_name.startswith("ENTREPRISE_"):
                print(f"Score: {score.item():.4f} -> {entity_name}")
else:
    print(f"Entity {target_entity_name} not found in the graph.")

1. Loading the Knowledge Graph into PyKEEN...
Graph loaded! Number of entities: 151, Number of relations: 6

2. Training the TransE Embedding Model (This may take 1-2 minutes)...


c:\Users\Maxime Philippon\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training epochs on cpu:   0%|          | 0/50 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/2.00 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/2.00 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/2.00 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/2.00 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/470 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.21s seconds



Training complete! Extracting embeddings...

3. Analyzing similarities based on graph structure...

--- Top 5 entities most similar to ENTREPRISE_302984158 ---
Score: 0.3523 -> ENTREPRISE_508175627
Score: 0.3379 -> ENTREPRISE_431899996
Score: 0.2940 -> ENTREPRISE_775685779
Score: 0.2795 -> ENTREPRISE_522026160
